# Empates duplos nas anotações totais

Este notebook replica a lógica de tratamento de empates do fase1_tratamento_logs e lista as frases com empate irresolúvel em ambos os grupos (masculino e feminino).

In [1]:
import pandas as pd
from itertools import combinations
from collections import Counter
from pathlib import Path

base = Path('data/logs_processados')
path_masc = base / 'MQD-7015_anotacoes_totais_masculinas.csv'
path_fem = base / 'MQD-6302_anotacoes_totais_femininas.csv'

def unresolved_detail(path):
    df = pd.read_csv(path, sep='\t')
    df['ordem_original'] = range(len(df))
    out_map = {}

    for frase, g in df.groupby('frase', sort=False):
        g = g.sort_values('ordem_original').reset_index(drop=True)
        n = len(g)

        if n < 4:
            out_map[frase] = 'menos_de_4'
            continue

        tie_pairs = Counter()
        has_plurality = False

        for idxs in combinations(range(n), 4):
            c = g.iloc[list(idxs)]['Value'].value_counts().to_dict()
            vals = sorted(c.values(), reverse=True)

            # Regra do notebook: há empate quando topo da contagem empata.
            if not (len(vals) >= 2 and vals[0] == vals[1]):
                has_plurality = True
                break

            if len(c) == 2 and set(c.values()) == {2}:
                tie_pairs[' vs '.join(sorted(c.keys()))] += 1
            else:
                tie_pairs['outro_empate'] += 1

        if not has_plurality:
            out_map[frase] = tie_pairs.most_common(1)[0][0] if tie_pairs else 'empate_irresolvel'

    return out_map

m = unresolved_detail(path_masc)
f = unresolved_detail(path_fem)
common = sorted(set(m) & set(f))

rows = []
for i, frase in enumerate(common, 1):
    rows.append({
        'id': i,
        'frase': frase,
        'tipo_empate_masculino': m[frase],
        'tipo_empate_feminino': f[frase],
        'mesmo_tipo_empate': m[frase] == f[frase],
    })

df_empates_duplos = pd.DataFrame(rows)
print(f'Total de empates duplos: {len(df_empates_duplos)}')
df_empates_duplos

Total de empates duplos: 19


,id,frase,tipo_empate_masculino,tipo_empate_feminino,mesmo_tipo_empate
0,1,"A senhora sempre respondia, vou bem graças a D...",neutra vs positiva,neutra vs positiva,True
1,2,Ao olhar novamente para a ponte viu o seu irmã...,neutra vs positiva,negativa vs neutra,False
2,3,Descobri essa irrefutável verdade ao perceber ...,neutra vs positiva,neutra vs positiva,True
3,4,"E foi daquelas colas de isopor mesmo, e já tav...",negativa vs neutra,negativa vs neutra,True
4,5,Ele não entende porque ele nunca teve que pass...,neutra vs positiva,neutra vs positiva,True
5,6,Essa é uma frase que eu costumo falar para alg...,negativa vs neutra,negativa vs neutra,True
6,7,"Eu cortei o cabelo, pintei, cortei, mudei minh...",negativa vs neutra,negativa vs neutra,True
7,8,"Eu encontro ele todo sábado, ele mora a duas o...",neutra vs positiva,neutra vs positiva,True
8,9,"Eu estou muito feliz com a aprovação, mesmo sa...",neutra vs positiva,neutra vs positiva,True
9,10,Eu não consigo acreditar que é tao difícil con...,neutra vs positiva,neutra vs positiva,True


In [2]:
output_csv = Path('data/resultados_gerais/empates_duplos_19_frases.csv')
output_csv.parent.mkdir(parents=True, exist_ok=True)
df_empates_duplos.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f'Arquivo salvo em: {output_csv}')

Arquivo salvo em: data\resultados_gerais\empates_duplos_19_frases.csv


In [3]:
# Resumos de frequência e matriz cruzada dos tipos de empate
freq_m = (
    df_empates_duplos['tipo_empate_masculino']
    .value_counts()
    .rename_axis('tipo_empate_masculino')
    .reset_index(name='quantidade')
)

freq_f = (
    df_empates_duplos['tipo_empate_feminino']
    .value_counts()
    .rename_axis('tipo_empate_feminino')
    .reset_index(name='quantidade')
)

matriz = pd.crosstab(
    df_empates_duplos['tipo_empate_masculino'],
    df_empates_duplos['tipo_empate_feminino']
).reset_index()

freq_m.to_csv('data/resultados_gerais/empates_duplos_freq_masculino.csv', index=False, encoding='utf-8-sig')
freq_f.to_csv('data/resultados_gerais/empates_duplos_freq_feminino.csv', index=False, encoding='utf-8-sig')
matriz.to_csv('data/resultados_gerais/empates_duplos_matriz_cruzada.csv', index=False, encoding='utf-8-sig')

print('Arquivos de resumo salvos em data/resultados_gerais/')
print('\nFrequência (masculino):')
display(freq_m)
print('\nFrequência (feminino):')
display(freq_f)
print('\nMatriz cruzada:')
display(matriz)

Arquivos de resumo salvos em data/resultados_gerais/

Frequência (masculino):


,tipo_empate_masculino,quantidade
0,neutra vs positiva,11
1,negativa vs neutra,7
2,negativa vs positiva,1



Frequência (feminino):


,tipo_empate_feminino,quantidade
0,neutra vs positiva,10
1,negativa vs neutra,9



Matriz cruzada:


tipo_empate_feminino,tipo_empate_masculino,negativa vs neutra,neutra vs positiva
0,negativa vs neutra,7,0
1,negativa vs positiva,0,1
2,neutra vs positiva,2,9
